In [23]:
%run sagemaker_utils.ipynb

  error: subprocess-exited-with-error
  
  × Building wheel for tokenizers (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [62 lines of output]
      /tmp/pip-build-env-asm3509b/overlay/lib/python3.11/site-packages/setuptools/dist.py:765: SetuptoolsDeprecationWarning: License classifiers are deprecated.
      !!
      
              ********************************************************************************
              Please consider removing the following classifiers in favor of a SPDX license expression:
      
              License :: OSI Approved :: Apache Software License
      
              See https://packaging.python.org/en/latest/guides/writing-pyproject-toml/#license for details.
              ********************************************************************************
      
      !!
        self._finalize_license_expression()
      running bdist_wheel
      running build
      running build_py
      creating build/lib.linux-x86_64-cpython-311/

In [ ]:
import json

# Bert TensorRT

In [24]:
!tar -C triton-serve-trt/ -czf model.tar.gz bert
model_uri = sagemaker_session.upload_data(path="model.tar.gz", key_prefix="triton-serve-pt")

In [25]:
sm_model_name = "triton-nlp-bert-trt-benchmark"
endpoint_config_name = "triton-nlp-bert-trt-benchmark"
endpoint_name = "triton-nlp-bert-trt-benchmark-2"

In [27]:
create_model(sm_model_name, model_uri, "bert")
create_endpoint_config(endpoint_config_name, sm_model_name)
create_endpoint(endpoint_name, endpoint_config_name)
poll(endpoint_name)

Model Arn: arn:aws:sagemaker:ap-south-1:978983596161:model/triton-nlp-bert-trt-benchmark


In [34]:
cleanup(sm_model_name, endpoint_config_name, endpoint_name)

# Bert Tokenizer (no model)

In [42]:
!tar -C triton-serve-tokenizer/ -czf model.tar.gz tokenizer
model_uri = sagemaker_session.upload_data(path="model.tar.gz", key_prefix="triton-serve-pt")

In [48]:
sm_model_name = "triton-nlp-tokenizer-benchmark"
endpoint_config_name = "triton-nlp-tokenizer-benchmark"
endpoint_name = "triton-nlp-tokenizer-benchmark-2"

In [44]:
create_model(sm_model_name, model_uri, "tokenizer")
create_endpoint_config(endpoint_config_name, sm_model_name)
create_endpoint(endpoint_name, endpoint_config_name)
poll(endpoint_name)

Model Arn: arn:aws:sagemaker:ap-south-1:978983596161:model/triton-nlp-tokenizer-benchmark
Endpoint Config Arn: arn:aws:sagemaker:ap-south-1:978983596161:endpoint-config/triton-nlp-tokenizer-benchmark
Endpoint Arn: arn:aws:sagemaker:ap-south-1:978983596161:endpoint/triton-nlp-tokenizer-benchmark-2
Status: Creating
Status: Creating
Status: Creating
Status: Creating
Status: Creating
Status: InService
Arn: arn:aws:sagemaker:ap-south-1:978983596161:endpoint/triton-nlp-tokenizer-benchmark-2
Status: InService


In [52]:
text_triton = "Triton Inference Server provides a cloud and edge inferencing solution optimized for both CPUs and GPUs."

payload = {
    "inputs": [
        {
            "name": "text",              # must match config.pbtxt
            "shape": [1, 1],
            "datatype": "BYTES",
            "data": [[text_triton]]
        }
    ]
}

response = client.invoke_endpoint(
    EndpointName=endpoint_name,
    ContentType="application/json",   # important change
    Body=json.dumps(payload)
)

result = json.loads(response["Body"].read().decode("utf8"))
print(result)

{'model_name': 'tokenizer', 'model_version': '1', 'outputs': [{'name': 'token_ids', 'datatype': 'INT32', 'shape': [1, 128], 'data': [101, 13012, 2669, 28937, 8241, 3640, 1037, 6112, 1998, 3341, 1999, 7512, 2368, 6129, 5576, 23569, 27605, 5422, 2005, 2119, 17368, 2015, 1998, 14246, 2271, 1012, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]}, {'name': 'attn_mask', 'datatype': 'INT32', 'shape': [1, 128], 'data': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0

In [53]:
cleanup(sm_model_name, endpoint_config_name, endpoint_name)

# bert TensorRT with tokenizer

In [59]:
!mkdir -p triton-serve-tokenizer-trt/bert_ensemble/1/
!tar -C triton-serve-tokenizer-trt/ -czf model.tar.gz bert bert_tokenizer bert_ensemble
model_uri = sagemaker_session.upload_data(path="model.tar.gz", key_prefix="triton-serve-pt")

In [63]:
sm_model_name = "triton-nlp-bert-with-tokenizer-benchmark"
endpoint_config_name = "triton-nlp-bert-with-tokenizer-benchmark"
endpoint_name = "triton-nlp-bert-with-tokenizer-benchmark-2"

In [64]:
create_model(sm_model_name, model_uri, "bert_ensemble")
create_endpoint_config(endpoint_config_name, sm_model_name)
create_endpoint(endpoint_name, endpoint_config_name)
poll(endpoint_name)

Model Arn: arn:aws:sagemaker:ap-south-1:978983596161:model/triton-nlp-bert-with-tokenizer-benchmark
Endpoint Config Arn: arn:aws:sagemaker:ap-south-1:978983596161:endpoint-config/triton-nlp-bert-with-tokenizer-benchmark
Endpoint Arn: arn:aws:sagemaker:ap-south-1:978983596161:endpoint/triton-nlp-bert-with-tokenizer-benchmark-2
Status: Creating
Status: Creating
Status: Creating
Status: Creating
Status: Creating
Status: InService
Arn: arn:aws:sagemaker:ap-south-1:978983596161:endpoint/triton-nlp-bert-with-tokenizer-benchmark-2
Status: InService


In [65]:
text_triton = "Triton Inference Server provides a cloud and edge inferencing solution optimized for both CPUs and GPUs."

payload = {
    "inputs": [
        {
            "name": "text",              # must match config.pbtxt
            "shape": [1, 1],
            "datatype": "BYTES",
            "data": [[text_triton]]
        }
    ]
}

response = client.invoke_endpoint(
    EndpointName=endpoint_name,
    ContentType="application/json",   # important change
    Body=json.dumps(payload)
)

result = json.loads(response["Body"].read().decode("utf8"))
print(result)

{'model_name': 'bert_ensemble', 'model_version': '1', 'parameters': {'sequence_id': 0, 'sequence_start': False, 'sequence_end': False}, 'outputs': [{'name': 'output', 'datatype': 'FP32', 'shape': [1, 768], 'data': [-0.9091796875, -0.257080078125, -0.06207275390625, 0.09710693359375, -0.304443359375, -0.02740478515625, -0.1175537109375, 0.2401123046875, -0.02569580078125, -0.447021484375, -0.09307861328125, -0.06280517578125, -0.300537109375, 0.55419921875, 0.299072265625, -0.1392822265625, -0.259033203125, 0.68310546875, 0.2310791015625, 0.19287109375, -0.58203125, -0.64013671875, 0.24658203125, -0.0279693603515625, -0.340576171875, 0.038848876953125, -0.2403564453125, -0.332275390625, 0.1861572265625, 0.30615234375, -0.48095703125, 0.27587890625, -0.17724609375, -0.74755859375, 0.7880859375, -0.354736328125, -0.261474609375, -0.1922607421875, -0.156005859375, 0.0264739990234375, -0.368408203125, -0.11480712890625, 0.374755859375, -0.1705322265625, 0.0379638671875, -0.06689453125, -3.1

In [66]:
cleanup(sm_model_name, endpoint_config_name, endpoint_name)

# Bert with IM requirement

In [67]:
!tar -C triton-serve-similarity/ -czf model.tar.gz bert bert_tokenizer cosine_similarity similarity_ensemble
model_uri = sagemaker_session.upload_data(path="model.tar.gz", key_prefix="triton-serve-pt")

In [68]:
sm_model_name = "triton-nlp-bert-with-imreq-benchmark"
endpoint_config_name = "triton-nlp-bert-with-imreq-benchmark"
endpoint_name = "triton-nlp-bert-with-imreq-benchmark"

In [69]:
create_model(sm_model_name, model_uri, "similarity_ensemble")
create_endpoint_config(endpoint_config_name, sm_model_name)
create_endpoint(endpoint_name, endpoint_config_name)
poll(endpoint_name)

Model Arn: arn:aws:sagemaker:ap-south-1:978983596161:model/triton-nlp-bert-with-imreq-benchmark
Endpoint Config Arn: arn:aws:sagemaker:ap-south-1:978983596161:endpoint-config/triton-nlp-bert-with-imreq-benchmark
Endpoint Arn: arn:aws:sagemaker:ap-south-1:978983596161:endpoint/triton-nlp-bert-with-imreq-benchmark
Status: Creating
Status: Creating
Status: Creating
Status: Creating
Status: Creating
Status: InService
Arn: arn:aws:sagemaker:ap-south-1:978983596161:endpoint/triton-nlp-bert-with-imreq-benchmark
Status: InService


In [71]:
query = "what is ai?"
answers = ["next token prediction", "it's not ai", "what is ai?", "openai google anthropic"]

payload = {
    "inputs": [
        {
            "name": "text",              # must match config.pbtxt
            "shape": [5, 1],
            "datatype": "BYTES",
            "data": [[query] + answers]
        }
    ]
}

response = client.invoke_endpoint(
    EndpointName=endpoint_name,
    ContentType="application/json",   # important change
    Body=json.dumps(payload)
)

result = json.loads(response["Body"].read().decode("utf8"))
print(result)

{'model_name': 'similarity_ensemble', 'model_version': '1', 'parameters': {'sequence_id': 0, 'sequence_start': False, 'sequence_end': False}, 'outputs': [{'name': 'scores', 'datatype': 'FP32', 'shape': [4], 'data': [0.8262747526168823, 0.9191592335700989, 0.9999999403953552, 0.8648236989974976]}]}


In [ ]:
cleanup(sm_model_name, endpoint_config_name, endpoint_name)